# Known-A ablation — final paired comparison

This notebook compares the frozen final learned-A results against the known-A
ablation for both single-shot and abundant-data regimes.

It is analysis-only and performs no training.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 280)

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "opinion_dynamics").exists():
            return p
    raise RuntimeError("Could not find repository root")

ROOT = find_repo_root()
R = ROOT / "opinion_dynamics" / "experiments" / "results"

SS_FINAL = R / "experiment_2026_09_17_broad_all_dynamics_multitopology_multiseed_unbounded_lambda1"
SS_KNOWN = R / "experiment_2026_09_19_broad_all_dynamics_multitopology_multiseed_knownA_unbounded_lambda1"
AB_FINAL = R / "experiment_2026_09_17_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1"
AB_KNOWN = R / "experiment_2026_09_19_abundant_all_dynamics_multitopology_multiseed_knownA_unbounded_lambda1"

for p in [SS_FINAL, SS_KNOWN, AB_FINAL, AB_KNOWN]:
    if not (p / "COMPUTATION_COMPLETE.json").exists():
        raise FileNotFoundError(f"Incomplete experiment: {p}")
    marker = json.loads((p / "COMPUTATION_COMPLETE.json").read_text(encoding="utf-8"))
    assert marker["status"] == "success"

print("All four computation markers are successful.")


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_59168\3859784416.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


All four computation markers are successful.


In [2]:
# Single-shot: read summaries from manifest-listed learned cache directories.
def load_single_shot(result_dir: Path, known_A: bool) -> pd.DataFrame:
    manifest = json.loads((result_dir / "RUN_MANIFEST.json").read_text(encoding="utf-8"))
    rows = []
    for env in manifest["environments"]:
        env_dir = Path(env["cache_dir"])
        for seed in manifest["learning_seeds"]:
            candidates = [d for d in (env_dir / "learned").glob(f"seed_{int(seed):02d}_*") if (d / "summary.json").exists()]
            if len(candidates) != 1:
                raise RuntimeError(f"Expected one learned cache for {env['environment_id']} seed {seed}; got {len(candidates)}")
            row = json.loads((candidates[0] / "summary.json").read_text(encoding="utf-8"))
            row["known_A"] = bool(known_A)
            rows.append(row)
    df = pd.DataFrame(rows)
    assert len(df) == 585
    return df

ss = pd.concat([
    load_single_shot(SS_FINAL, False),
    load_single_shot(SS_KNOWN, True),
], ignore_index=True)

# Abundant: combined summaries are already merged by the experiment notebook.
ab_final = pd.read_csv(AB_FINAL / "combined_summary.csv")
ab_known = pd.read_csv(AB_KNOWN / "combined_summary.csv")
ab_final = ab_final[ab_final["learning_seed"].notna()].copy()
ab_known = ab_known[ab_known["learning_seed"].notna()].copy()
ab_final["known_A"] = False
ab_known["known_A"] = True
ab = pd.concat([ab_final, ab_known], ignore_index=True)
assert len(ab_final) == len(ab_known) == 585

print("Single-shot rows:", len(ss))
print("Abundant rows:", len(ab))


Single-shot rows: 1170
Abundant rows: 1170


In [3]:
def env_means(df, performance_col):
    return (
        df.groupby(["known_A", "environment_id", "scenario_class", "family", "dynamics"], dropna=False, as_index=False)
        .agg(performance=(performance_col, "mean"), prediction_ratio=("final_fit_model_over_identity" if "final_fit_model_over_identity" in df.columns else "model_over_identity", "mean"))
    )

ss_env = env_means(ss, "final_mean")
ab_env = env_means(ab, "mean_end")

def paired_summary(env_df, regime):
    learned = env_df[~env_df["known_A"]].drop(columns="known_A")
    known = env_df[env_df["known_A"]].drop(columns="known_A")
    p = learned.merge(known, on=["environment_id", "scenario_class", "family", "dynamics"], suffixes=("_learnedA", "_knownA"), validate="one_to_one")
    p["delta_performance_knownA_minus_learnedA"] = p["performance_knownA"] - p["performance_learnedA"]
    p["delta_prediction_ratio_knownA_minus_learnedA"] = p["prediction_ratio_knownA"] - p["prediction_ratio_learnedA"]
    rows=[]
    for (scenario,dyn),g in p.groupby(["scenario_class","dynamics"], sort=False):
        x=g["delta_performance_knownA_minus_learnedA"].to_numpy(float)
        mean=float(x.mean()); n=len(x)
        if n>1:
            half=float(student_t.ppf(0.975,n-1)*x.std(ddof=1)/np.sqrt(n))
        else:
            half=np.nan
        rows.append({
            "regime":regime,
            "scenario_class":scenario,
            "dynamics":dyn,
            "n_environments":n,
            "learnedA_performance_mean":float(g["performance_learnedA"].mean()),
            "knownA_performance_mean":float(g["performance_knownA"].mean()),
            "knownA_minus_learnedA_mean":mean,
            "knownA_minus_learnedA_ci_low":mean-half,
            "knownA_minus_learnedA_ci_high":mean+half,
            "knownA_win_rate":float((x>0).mean()),
            "learnedA_prediction_ratio_mean":float(g["prediction_ratio_learnedA"].mean()),
            "knownA_prediction_ratio_mean":float(g["prediction_ratio_knownA"].mean()),
        })
    return p, pd.DataFrame(rows)

ss_paired, ss_summary = paired_summary(ss_env, "single_shot")
ab_paired, ab_summary = paired_summary(ab_env, "abundant")
summary = pd.concat([ss_summary, ab_summary], ignore_index=True)
display(summary.round(5))


,regime,scenario_class,dynamics,n_environments,learnedA_performance_mean,knownA_performance_mean,knownA_minus_learnedA_mean,knownA_minus_learnedA_ci_low,knownA_minus_learnedA_ci_high,knownA_win_rate,learnedA_prediction_ratio_mean,knownA_prediction_ratio_mean
0,single_shot,structured_mechanism,friedkinjohnsen,15,0.61128,0.61384,0.00256,0.00084,0.00429,0.93333,0.06727,0.07050
1,single_shot,structured_mechanism,hegselmannkrause,15,0.87643,0.89532,0.01889,0.01757,0.02021,1.00000,0.07497,0.06256
2,single_shot,structured_mechanism,nonlinearinfluence,15,0.86499,0.86557,0.00058,0.00022,0.00093,0.93333,0.03377,0.03310
3,single_shot,unstructured_random,coca,25,0.76397,0.78027,0.01630,0.00531,0.02729,0.76000,0.22309,0.13757
4,single_shot,unstructured_random,friedkinjohnsen,25,0.68442,0.69559,0.01117,0.00775,0.01459,1.00000,0.16166,0.22625
5,single_shot,unstructured_random,hegselmannkrause,25,0.79023,0.84151,0.05128,0.03639,0.06616,1.00000,0.23297,0.11510
6,single_shot,unstructured_random,laplacian,25,0.76363,0.81652,0.05289,0.04108,0.06469,1.00000,0.10324,0.02835
7,single_shot,unstructured_random,nonlinearinfluence,25,0.82011,0.84478,0.02467,0.01981,0.02953,1.00000,0.06670,0.01956
8,single_shot,unstructured_random,repulsion,25,0.72312,0.73616,0.01304,-0.00759,0.03366,0.52000,0.46392,0.50904
9,abundant,structured_mechanism,friedkinjohnsen,15,0.61367,0.61384,0.00017,-0.00001,0.00034,0.60000,0.30314,0.09061


In [4]:
out = R / "known_A_ablation_analysis_2026_09_19"
out.mkdir(parents=True, exist_ok=True)
summary.to_csv(out / "known_A_ablation_summary_by_regime_dynamics.csv", index=False)
ss_paired.to_csv(out / "known_A_ablation_single_shot_environment_level.csv", index=False)
ab_paired.to_csv(out / "known_A_ablation_abundant_environment_level.csv", index=False)
print("Saved:", out)
print("Analysis complete. Positive knownA-minus-learnedA means fixing A_true improved control.")


Saved: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\known_A_ablation_analysis_2026_09_19
Analysis complete. Positive knownA-minus-learnedA means fixing A_true improved control.
